# end to end template — exercises

10 tasks that test what the cheat sheet `03_end_to_end_template.ipynb` covers. Try each one
**before** looking at the cheat sheet.

How it works:
1. Read the task. Write your code in the cell below it, assigning the result to the
   variable the task names (usually `answer`).
2. Run the cell: the last line `qN.check()` tells you ✅ or ❌ with a short reason.
3. Stuck? Uncomment `qN.hint()` for a nudge, or `qN.solution()` to see the reference code.
These tasks walk through the raw hourly file `hourly_power_raw.csv` in the order you would in the interview: inspect, clean, define, feature, split, baseline, model, diagnose.

In [ ]:
import sys; sys.path.append("..")
from quantlearn import load_questions
q1, q2, q3, q4, q5, q6, q7, q8, q9, q10 = load_questions("05_time_series_research/03_end_to_end_template")

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

raw = pd.read_csv("../data/hourly_power_raw.csv")

## Task 1

Parse the `time` column of `raw` as UTC timestamps and assign the number of **duplicated timestamps** (rows whose timestamp already appeared earlier).

Assign the result to `answer`.

In [ ]:
# your solution here
answer = ____

q1.check()

In [ ]:
# q1.hint()
# q1.solution()

## Task 2

Assign the number of `temp_c` values equal to the sentinel -999.

Assign the result to `answer`.

In [ ]:
# your solution here
answer = ____

q2.check()

In [ ]:
# q2.hint()
# q2.solution()

## Task 3

`price_eur_mwh` is text. Convert it to numbers so that bad values become NaN, and assign how many NaN that produces.

Assign the result to `answer`.

In [ ]:
# your solution here
answer = ____

q3.check()

In [ ]:
# q3.hint()
# q3.solution()

## Task 4

Sort by time, drop duplicated timestamps (keep the last), set `time` as the index, reindex to the complete hourly grid from the first to the last timestamp, and assign the number of hours whose `consumption_mwh` is missing.

Assign the result to `answer`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)

# your solution here
answer = ____

q4.check()

In [ ]:
# q4.hint()
# q4.solution()

## Task 5

`df` is the cleaned hourly frame on the full grid. The problem: at hour t predict consumption at t+24. Build `d` with columns `y` = consumption 24 h ahead, `lag24`, `lag168`, `roll24` = mean of the 24 hours before t (shift first), `temp_c`, `hour`, `dow`; drop rows with any NaN once. Assign the first timestamp of `d`.

Assign the result to `answer`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)
df = raw.assign(time=t).sort_values("time").drop_duplicates("time", keep="last").set_index("time")
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_idx)
df.index.name = "time"

# your solution here
answer = ____

q5.check()

In [ ]:
# q5.hint()
# q5.solution()

## Task 6

Chronological split: the first 80% of the rows of `d` (`split = int(len(d) * 0.8)`) are train, the rest test. Assign `n_test` = number of test rows and `naive_rmse` = RMSE on the test rows of the naive forecast `lag24` against `y`.

Assign the result to `n_test`, `naive_rmse`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)
df = raw.assign(time=t).sort_values("time").drop_duplicates("time", keep="last").set_index("time")
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_idx)
df.index.name = "time"

d = pd.DataFrame(index=df.index)
d["y"] = df["consumption_mwh"].shift(-24)
d["lag24"] = df["consumption_mwh"].shift(24)
d["lag168"] = df["consumption_mwh"].shift(168)
d["roll24"] = df["consumption_mwh"].shift(1).rolling(24).mean()
d["temp_c"] = df["temp_c"]
d["hour"] = d.index.hour
d["dow"] = d.index.dayofweek
d = d.dropna()
split = int(len(d) * 0.8)
train, test = d.iloc[:split], d.iloc[split:]
features = ["lag24", "lag168", "roll24", "temp_c", "hour", "dow"]

# your solution here
n_test = ____
naive_rmse = ____

q6.check()

In [ ]:
# q6.hint()
# q6.solution()

## Task 7

Fit `Ridge(alpha=1.0)` on the train rows using `features`, predict the test rows, and assign the test RMSE.

Assign the result to `answer`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)
df = raw.assign(time=t).sort_values("time").drop_duplicates("time", keep="last").set_index("time")
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_idx)
df.index.name = "time"

d = pd.DataFrame(index=df.index)
d["y"] = df["consumption_mwh"].shift(-24)
d["lag24"] = df["consumption_mwh"].shift(24)
d["lag168"] = df["consumption_mwh"].shift(168)
d["roll24"] = df["consumption_mwh"].shift(1).rolling(24).mean()
d["temp_c"] = df["temp_c"]
d["hour"] = d.index.hour
d["dow"] = d.index.dayofweek
d = d.dropna()
split = int(len(d) * 0.8)
train, test = d.iloc[:split], d.iloc[split:]
features = ["lag24", "lag168", "roll24", "temp_c", "hour", "dow"]

# your solution here
answer = ____

q7.check()

In [ ]:
# q7.hint()
# q7.solution()

## Task 8

Using the fitted Ridge model `model` and its test predictions `pred`, assign the **hour of day** (0-23) with the largest mean absolute error on the test set.

Assign the result to `answer`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)
df = raw.assign(time=t).sort_values("time").drop_duplicates("time", keep="last").set_index("time")
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_idx)
df.index.name = "time"

d = pd.DataFrame(index=df.index)
d["y"] = df["consumption_mwh"].shift(-24)
d["lag24"] = df["consumption_mwh"].shift(24)
d["lag168"] = df["consumption_mwh"].shift(168)
d["roll24"] = df["consumption_mwh"].shift(1).rolling(24).mean()
d["temp_c"] = df["temp_c"]
d["hour"] = d.index.hour
d["dow"] = d.index.dayofweek
d = d.dropna()
split = int(len(d) * 0.8)
train, test = d.iloc[:split], d.iloc[split:]
features = ["lag24", "lag168", "roll24", "temp_c", "hour", "dow"]
model = Ridge(alpha=1.0).fit(train[features], train["y"])
pred = model.predict(test[features])

# your solution here
answer = ____

q8.check()

In [ ]:
# q8.hint()
# q8.solution()

## Task 9

Assign the name (string) of the feature with the largest **absolute** Ridge coefficient.

Assign the result to `answer`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)
df = raw.assign(time=t).sort_values("time").drop_duplicates("time", keep="last").set_index("time")
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_idx)
df.index.name = "time"

d = pd.DataFrame(index=df.index)
d["y"] = df["consumption_mwh"].shift(-24)
d["lag24"] = df["consumption_mwh"].shift(24)
d["lag168"] = df["consumption_mwh"].shift(168)
d["roll24"] = df["consumption_mwh"].shift(1).rolling(24).mean()
d["temp_c"] = df["temp_c"]
d["hour"] = d.index.hour
d["dow"] = d.index.dayofweek
d = d.dropna()
split = int(len(d) * 0.8)
train, test = d.iloc[:split], d.iloc[split:]
features = ["lag24", "lag168", "roll24", "temp_c", "hour", "dow"]
model = Ridge(alpha=1.0).fit(train[features], train["y"])

# your solution here
answer = ____

q9.check()

In [ ]:
# q9.hint()
# q9.solution()

## Task 10

Assign the lag-1 autocorrelation of the test residuals (`y − pred`) using pandas `autocorr(1)`.

Assign the result to `answer`.

In [ ]:
t = pd.to_datetime(raw["time"], utc=True)
df = raw.assign(time=t).sort_values("time").drop_duplicates("time", keep="last").set_index("time")
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_idx)
df.index.name = "time"

d = pd.DataFrame(index=df.index)
d["y"] = df["consumption_mwh"].shift(-24)
d["lag24"] = df["consumption_mwh"].shift(24)
d["lag168"] = df["consumption_mwh"].shift(168)
d["roll24"] = df["consumption_mwh"].shift(1).rolling(24).mean()
d["temp_c"] = df["temp_c"]
d["hour"] = d.index.hour
d["dow"] = d.index.dayofweek
d = d.dropna()
split = int(len(d) * 0.8)
train, test = d.iloc[:split], d.iloc[split:]
features = ["lag24", "lag168", "roll24", "temp_c", "hour", "dow"]
model = Ridge(alpha=1.0).fit(train[features], train["y"])
pred = model.predict(test[features])

# your solution here
answer = ____

q10.check()

In [ ]:
# q10.hint()
# q10.solution()

---
Done? Re-open the cheat sheet for anything you had to look up, then try the next `_tests` notebook.